----
## <font color='CornflowerBlue'>Practical 6: Detecting uncertainty in generated answers</font> 
##### Marieke Smith, Alok Bharadwaj, and Arjen Jakobi
---

### Semantic Entropy: Detecting uncertainty in generated answers

In the previous notebooks, we studied uncertainty at the level of **tokens**: how much probability the model assigns to one answer option or to the next generated word. In this notebook, we move one level higher. We will study uncertainty at the level of **meaning**.

This matters because language models can express the same idea in many different ways. If we ask the same question multiple times, the model might produce answers that use different words but still mean the same thing. Token-level variation alone is therefore not enough to measure uncertainty. Instead, we want to ask whether the model keeps returning the same underlying claim, or whether its answers differ in meaning.

The method we use is called **semantic entropy**. The idea is to sample multiple answers from the model, group answers that mean the same thing using an entailment model, and measure how spread out the answers are across different meaning clusters. Low semantic entropy means that the model repeatedly gives semantically similar answers. High semantic entropy means that the model gives answers with different meanings, which may indicate uncertainty and a higher risk of confabulation.

In this exercise, we adapt the semantic entropy pipeline so that we can colour-code parts of a generated answer. We first generate a three-sentence answer, split it into smaller sentence parts, and extract factual claims, or **factoids**, from each part. For each factoid, we ask the model several related questions and sample multiple answers. We then use semantic entropy to estimate how consistently the model supports that factoid.

The pipeline is:

1. Generate a three-sentence answer to a user-defined question.
2. Split the answer into sentence parts, with at most two parts per sentence.
3. Extract one or two factoids from each sentence part.
4. Turn each factoid into `Q` related questions.
5. Ask the model to answer each question `M` times.
6. Use an entailment model to group semantically equivalent answers.
7. Compute semantic entropy for each question.
8. Average across questions to obtain a semantic entropy score for each factoid.
9. Convert factoid-level entropy into sentence-part confidence.
10. Colour-code the original answer based on semantic confidence.

The goal is not only to implement the pipeline, but also to understand where it works and where it can fail. Several steps rely on prompts and model-generated intermediate outputs, so you should expect some noise. Part of the exercise is to inspect these intermediate outputs critically.  For most steps the code will be provided, you can just run it and analyze what is happening.

Most of the code is provided for you. You will write or complete code in Question 3, Question 4, and Question 5. These correspond to pipeline steps 4, 5, and the combined semantic-entropy computation over steps 6–8.

#### Before You Start
Run the setup cells below to load the required packages, download the entailment model, and create the Hugging Face inference client. You do not need to understand every line of the helper code, but you should understand the role each component plays in the pipeline.

#### Further reading
The general idea of this method was discussed in the lecture, but for an overview you can also read this blogpost: https://oatml.cs.ox.ac.uk/blog/2024/06/19/detecting_hallucinations_2024.html 

In [ ]:
#@title Click here to initialize the environment (Required) { display-mode: "form" }
import os, sys, subprocess

print("Initializing environment. Please wait.")

# Clone the git repo 
subprocess.run(["git", "clone", "https://github.com/cryoTUD/ai4nanobiology.git"], stdout=subprocess.DEVNULL)
os.chdir("ai4nanobiology/week_6")
sys.path.append(os.path.abspath('.'))

# Installing requirements
subprocess.run(["pip", "install", "-q", "sentencepiece", "protobuf", "tiktoken", "transformers", "accelerate"])
print("Environment ready! You can now start the exercise.")

### Setup: Import required packages

In [1]:
import requests
import numpy as np
import pandas as pd
import re
import math
import json
import torch
import re
import time

from src.utils import (
    setup_client, 
    hex_to_rgb, 
    colored, 
    prob_to_color,
    query_model,
)


### Setup: Load the entailment model and inference client

Semantic entropy requires two types of models. First, we use a language model to generate answers, questions, and repeated samples. Second, we use an **entailment model** to decide whether two answers express the same meaning.

The next cells import the semantic entropy utilities, load the entailment model, and set up the Hugging Face client used to query the language model.

In [ ]:
from src.semantic_entropy_utils import (
    EntailmentDeberta,
    get_semantic_ids,
    cluster_assignment_entropy
)

In [ ]:
entailment_model = EntailmentDeberta()

In [ ]:
hf_client = setup_client()

## 1. Generate an initial answer

We begin by asking the language model to produce a short answer to a question. To keep the later analysis manageable, the prompt asks for exactly three sentences and avoids bullet points.

This initial answer is the object we will later analyse and colour-code. Try to choose a question for which you can judge whether the answer is plausible. The method is most informative when the model may know some relevant information, but may also be uncertain or tempted to invent details.

In [ ]:
# initialize global variables
model_name = "llama-8b"
max_tokens = 1000
client = hf_client

In [ ]:
def generate_long_answer(question, model_name, max_tokens, client):
    '''
    Generates a longer answer of three sentences to the provided main question. 
    Returns a string that contains the answer.
    '''

    prompt = question + '\nAnswer in exactly 3 sentences. Do not use bullet points in your answer.'
    response = query_model('', prompt, model_name, max_tokens, client)
    answer = response['answer']
    return answer


### Try a question of your own

Run the next cell to generate an answer. You can replace the example question with one related to a topic of your own interest.

A good question for this exercise has two properties:

- it asks for factual information rather than an opinion;
- you know enough about the topic to recognise possible mistakes.

This makes it easier to compare your own judgement with the semantic entropy scores later in the notebook.

In [ ]:
question = "Can you give me examples of student rowing associations in Delft?" # TO DO: change to something of your interest
answer = generate_long_answer(question, model_name, max_tokens, client)
print(answer)

### Question 1: First fact check

Before using semantic entropy, inspect the model’s answer yourself.

Which parts of the answer seem correct? Which parts seem uncertain, unsupported, or likely to be false? Briefly explain your reasoning. This manual judgement will serve as a reference point when you later interpret the colour-coded output.

In [ ]:
# TO DO: write your answer here


## 2. Split the answer and extract factoids

The next part of the pipeline prepares the generated answer for semantic analysis. Instead of assigning one uncertainty score to the whole answer, we split the answer into smaller **sentence parts** and extract factual claims from each part.

A **factoid** is a short factual claim that can be checked or challenged independently. For example, from the sentence part “Marie Curie moved to Paris for her studies”, we might extract the factoid “Marie Curie moved to Paris” or “Marie Curie moved to Paris for her studies”.

Read the docstrings and prompts in the next two functions:

- `extract_raw_json_pipeline(...)` uses the language model to split the answer and extract factoids.
- `parse_json_to_dictionaries(...)` converts the model output into Python dictionaries used later in the notebook.

You do not need to understand every implementation detail, but you should understand the input and output of each function.

In [ ]:
def extract_raw_json_pipeline(answer, model_name, max_tokens, client):
    """
    Makes 1 API call to split the answer into sentences, then 1 API call per sentence 
    to extract sentence parts and factoids
    Returns raw parsed JSON
    """
    print("=" * 80)
    print("SENTENCE SPLITTING (JSON)")
    print("=" * 80)
    
    sys_prompt_1 = "You are a strict text parser. You ONLY output valid JSON arrays of strings. No markdown, no explanations."
    user_prompt_1 = f"""Split the following text into individual sentences.
    Rules:
    1. Output a maximum of 3 sentences.
    2. Respond ONLY with a valid JSON array of strings.
    
    Text: "{answer}"
    
    Example Output:
    ["Sentence 1.", "Sentence 2.", "Sentence 3."]
    """
    
    # First API call: split the sentences using LLM
    try:
        res_sentences = query_model(sys_prompt_1, user_prompt_1, model_name, max_tokens, client)
        raw_sentences = res_sentences.get('answer', '').strip()
        clean_json_str = re.sub(r'```json|```', '', raw_sentences).strip() # safety cleaner to process potential "```json" in answer
        sentences = json.loads(clean_json_str)
        print(f"Successfully extracted {len(sentences)} sentences via JSON.")
    except Exception as e:
        print(f"❌ Error during sentence splitting: {e}")
        return [], []

    print("\n" + "=" * 80)
    print("FACTOID EXTRACTION (JSON)")
    print("=" * 80)
    
    master_json_output = []
    sys_prompt_2 = "You are a strict data extraction tool. You ONLY output valid JSON arrays of objects. You NEVER invent external information."
    
    # API calls 2-4: split into 2 sentence parts, and split those into 2 factoids
    for i, sentence in enumerate(sentences, 1):
        print(f"Processing Sentence {i}/{len(sentences)}: '{sentence}'")
        
        user_prompt_2 = f"""Decompose the given sentence into consecutive 'sentence parts' and extract factoids.
                
        RULES:
        1. Min 1, max 2 sentence parts total.
        2. EXACT COPY: The 'sentence part' MUST be an exact, unaltered substring copied directly from the original sentence.
        3. Min 1, max 2 brief factoids per sentence part.
        4. STRICT: ONLY extract facts explicitly stated.
        5. Respond ONLY with a valid JSON array.
        6. Be exact in the factoids that you generate: provide names of the people/places/occurences that you mention.
        
        EXAMPLE INPUT:
        Sentence: "Marie Curie was born in Warsaw, and she later moved to Paris to study."
        
        EXAMPLE OUTPUT:
        [
          {{
            "sentence_part": "Marie Curie was born in Warsaw, ",
            "factoids": ["Marie Curie was born in Warsaw"]
          }},
          {{
            "sentence_part": "and she later moved to Paris to study.",
            "factoids": ["Marie Curie moved to Paris", "Marie Curie moved to Paris for her studies"]
          }}
        ]
        
        ACTUAL INPUT:
        Sentence: "{sentence}"
        """
        
        try:
            res_factoids = query_model(sys_prompt_2, user_prompt_2, model_name, max_tokens, client)
            raw_output = res_factoids.get('answer', '').strip()
            clean_output = re.sub(r'```json|```', '', raw_output).strip() # again safety check to remove additional json stuff
            
            # Parse the JSON and append to master list
            parsed_sentence_data = json.loads(clean_output) # json.loads() parses the json string and puts the contents into a dictionary
            master_json_output.append(parsed_sentence_data) 
            
        except Exception as e:
            print(f"❌ Error parsing JSON for sentence {i}: {e}")
            master_json_output.append([]) # Append empty list on failure to keep index aligned
            
    return master_json_output, sentences

In [ ]:
def parse_json_to_dictionaries(master_json_output, original_sentences):
    """
    Takes the raw JSON output from the LLM pipeline, applies Python code with safety limits,
    and puts it into the designated dictionaries
    """
    sentence_part_to_factoids = {}
    factoid_to_sentence_part = {}
    sentence_parts = []
    factoids_list = []
    
    for sentence_data, original_sentence in zip(master_json_output, original_sentences):
        constrained_parts = sentence_data[:2] # ensure there are only two sentence parts
        
        for item in constrained_parts:
            s_part = item.get("sentence_part", "").strip() # get the sentence part
            extracted_factoids = item.get("factoids", []) # get the corresponding factoids
            
            if not s_part:
                continue
                
            if s_part not in original_sentence:
                print(f"⚠️ WARNING: Paraphrase detected! \nExpected substring of: '{original_sentence}' \nGot: '{s_part}'")
            
            constrained_factoids = [f.strip() for f in extracted_factoids if f.strip()][:2] # strip of whitespace & ensure there are only two factoids

            # print warning if a sentence part has no extracted factiods
            if len(constrained_factoids) == 0:
                print(f"⚠️ Notice: No valid factoids extracted for sentence part: '{s_part}'")
                continue

            # store the sentence part and the factoids in the corresponding datastructures
            sentence_parts.append(s_part) 
            sentence_part_to_factoids[s_part] = constrained_factoids
            
            for factoid in constrained_factoids:
                factoids_list.append(factoid)
                factoid_to_sentence_part[factoid] = s_part

    return sentence_part_to_factoids, factoid_to_sentence_part, sentence_parts, factoids_list

#### Question 2: Inspect the parsed data structure

Run the next cell to execute the sentence splitting and factoid-extraction pipeline.

This step is deliberately worth inspecting carefully. We are using a relatively small language model as a parser, so the output may not always follow the instructions perfectly. You may need to run the cell more than once and keep an iteration that is useful for the rest of the exercise. Aim for an output with at least two sentence parts and at least four extracted factoids in total.

After running the cell, examine the final parsed data structure. Do the sentence parts correspond to meaningful fragments of the original answer? Are the extracted factoids clear, factual, and specific? Note any caveats, such as missing claims, duplicated factoids, paraphrased sentence parts, or claims that are too vague to verify.

In [ ]:
# Run the API calls
print("-" * 80 + f'\nORIGINAL ANSWER WAS:\n {answer}\n' + "-"*80)

raw_json_data, original_sentences = extract_raw_json_pipeline(answer, model_name, max_tokens, client)

# Parse the json output
s_part_to_factoids, factoid_to_s_part, sentence_parts, factoids = parse_json_to_dictionaries(raw_json_data, original_sentences)

print("\n" + "=" * 80)
print("FINAL PARSED DATA STRUCTURE")
print("=" * 80)
for i, s_part in enumerate(sentence_parts, 1):
    print(f"[{i}] '{s_part}'")
    for factoid in s_part_to_factoids[s_part]:
        print(f"    -> {factoid}")

In [ ]:
# TO DO: write your answer here

### Why Factoids Matter

The quality of the extracted factoids strongly affects the rest of the pipeline. If a factoid is too broad, several different answers may all seem partly correct. If it is too narrow or poorly phrased, the generated questions may not target the intended claim. Treat the parsed structure as an intermediate result that needs human inspection, not as ground truth.


## 3. Turn each factoid into `Q` questions

To estimate whether the model is semantically confident about a factoid, we do not only ask the model to repeat the factoid. Instead, we generate several questions that the factoid should answer.

For example:

**Input factoid**

```text
Marie Curie moved to Paris for her studies
```

**Possible output when `Q = 2`**

```python
["Why did Marie Curie move to Paris?", "Where did Marie Curie move to for her studies?"]
```

The purpose of this step is to probe the same factual claim from slightly different angles. If the factoid is well supported by the model, the model should give semantically consistent answers to these related questions.

### Question 3: Write a prompt for question generation

Edit the `generate_Q_questions(...)` function by writing a suitable user prompt. Your prompt may use the variables `Q`, `original_answer`, and `factoid` inside the f-string.

A good prompt should include clear rules. For example, it should tell the model to:

- generate exactly `Q` questions;
- make each question answerable by the factoid;
- avoid asking about information that is not present in the factoid;

It is often helpful to include an example input and example output. After writing your prompt, run the question-generation cell and inspect the output. Improve the prompt if the generated questions are ambiguous, too broad, or not directly answered by the factoid. Paste your final prompt in the cell below.

In [ ]:
## TO DO: paste your final user prompt here
user_prompt = 

In [ ]:
def generate_Q_questions(factoid, original_answer, Q, model_name, max_tokens, client, enable_sleep=True):
    '''
    Generates Q questions that might have generated the factoid, using the original answer for context.
    
    Args:
        - factoid: string, the current factoid to turn into questions
        - original_answer: string, the original full text
        - Q: int, number of questions to generate
        - model_name: string, name of the model
        - max_tokens: int
        - client: the proxy URL
        - enable_sleep: boolean, enables sleep function if rate limit is hit
        
    Returns:
        - questions: a list of exactly Q question strings
    '''
       
    sys_prompt = """You are a precise data generator. You ONLY output valid JSON arrays of strings. No explanations, no markdown, no conversational text."""
    
    user_prompt = f"""   """ # TO DO: write your user prompt here
    
    # Retry loop: query the model and if the rate limit is hit, wait for 20 secs with a maximum of 3 retries
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = query_model(sys_prompt, user_prompt, model_name, max_tokens, client)
            raw_output = response.get('answer', '').strip() # extract and strip the answer
            clean_json_str = re.sub(r'```json|```', '', raw_output).strip() # remove potential json mark
            questions = json.loads(clean_json_str) # load the answer into a dictionary
            constrained_questions = [str(q).strip() for q in questions][:Q] # strip and ensure that there are max Q questions
            return constrained_questions
            
        except Exception as e:
            error_str = str(e).lower()

            # if the rate limit is hit, sleep for 20 secs
            if enable_sleep and ('429' in error_str or 'rate limit' in error_str or 'too many requests' in error_str):
                wait_time = 20 # Wait 20 seconds before trying again
                print(f"⏳ Rate limit hit! Sleeping for {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"❌ ERROR: API Error for factoid '{factoid[:30]}...': {e}")
                if attempt == max_retries - 1 or not enable_sleep:
                    return []
    return []

In [ ]:
print("=" * 80)
print(f"GENERATING Q QUESTIONS PER FACTOID")
print("=" * 80)

Q = 2 # Number of questions per factoid
ENABLE_SLEEP = True # if set to true, the function pauses for 20 seconds when the rate limit is hit
factoid_to_questions = {}

for i, factoid in enumerate(factoids, 1):
    print(f"\nProcessing Factoid {i}/{len(factoids)}: '{factoid}'")
    
    # Generate the questions
    sub_questions = generate_Q_questions(
        factoid=factoid, 
        original_answer=answer, 
        Q=Q, 
        model_name=model_name, 
        max_tokens=max_tokens, 
        client=client,
        enable_sleep=ENABLE_SLEEP 
    )
    
    # Store in our dictionary
    factoid_to_questions[factoid] = sub_questions
    
    # Print to check what happened
    if not sub_questions:
        print("  FAILED to generate questions.")
    else:
        for j, q in enumerate(sub_questions, 1):
            print(f"  -> Q{j}: {q}")

print("\n" + "=" * 80)
print("SUCCESS: Question generation complete!")

### Interpreting the generated questions
Before moving on, check whether the generated questions really test the intended factoids. A poor question can make the entropy score misleading. For example, a question that is too general may allow several correct answers, while a question that adds information not present in the factoid may test a different claim altogether.

## 4. Answer Each Question `M` Times

Next, we ask the model to answer each generated question multiple times. These repeated samples reveal whether the model consistently gives the same semantic answer or whether it alternates between different meanings.

The answers should be short and easy to compare. Long answers introduce extra wording that can make the entailment step noisier. Ideally, each answer should contain only the direct answer entity or short phrase.

#### Question 4: Write a prompt for repeated answer generation

Now write the user prompt for `generate_M_answers(...)`. The goal is different from the previous step: here, the model should answer a single question concisely, and it should do so in a format that is easy for the entailment model to compare.

Your prompt should include clear rules. For example, ask the model to:

- answer only the question provided;
- avoid full sentences when a short phrase is enough;
- avoid explanations or conversational text;
- return one direct answer per query.

Run the cell that calls the function and inspect the sampled answers. If the answers are too long, inconsistent in format, or include unnecessary explanation, revise your prompt. Paste your final prompt in the cell below.

In [ ]:
# TO DO: my final user_prompt
user_prompt = 

In [ ]:
def generate_M_answers(question, M, model_name, max_tokens, client, enable_sleep=True):
    '''
    Provides M independent answers to the current question to measure model uncertainty.
    
    Args:
        - question: string, the question to answer
        - M: int, number of times to ask the question
        - model_name: string, name of the model
        - max_tokens: int
        - client: the proxy URL
        - enable_sleep: boolean, enables sleep function if rate limit is hit
        
    Returns:
        - sub_answers: list of M answer strings
    '''
    
    sys_prompt = "You are a highly concise question-answering system. You ONLY output the direct answer entity. No full sentences, no conversational text."
    user_prompt = f"""   """ # TO DO: your user prompt here
    sub_answers = []
    
    for i in range(M):
        # Again enable 3 retries if rate limit is hit 
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = query_model(sys_prompt, user_prompt, model_name, max_tokens, client)
                raw_answer = response.get('answer', '').strip()
                sub_answers.append(raw_answer)
                break # If a response is generated, break out of the retry loop and go to the next 'M'
            except Exception as e:
                error_str = str(e).lower()
                if enable_sleep and ('429' in error_str or 'rate limit' in error_str or 'too many requests' in error_str):
                    wait_time = 20
                    print(f"⏳ Rate limit hit! Sleeping for {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                else:
                    print(f"❌ Error during API call {i+1} for question: {e}")
                    if attempt == max_retries - 1 or not enable_sleep:
                        sub_answers.append("ERROR") 
                        break
    return sub_answers

In [ ]:
print("=" * 80)
print(f"ANSWERING EACH QUESTION M TIMES")
print("=" * 80)

M_answers = 3 # Let's start with 3 to save API calls during testing
ENABLE_SLEEP = True 
question_to_answers = {}

# Iterate through every factoid, and then through every question for that factoid
for factoid, questions in factoid_to_questions.items():
    for q_idx, question in enumerate(questions, 1):
        print(f"\nProcessing Question: '{question}'")
        
        # Get M independent answers
        answers = generate_M_answers(
            question=question, 
            M=M_answers, 
            model_name=model_name, 
            max_tokens=max_tokens, 
            client=client,
            enable_sleep=ENABLE_SLEEP 
        )
        
        # Store in our new dictionary
        question_to_answers[question] = answers
        
        # Print the M answers 
        for m_idx, ans in enumerate(answers, 1):
            print(f"  ↳ Answer {m_idx}: {ans}")

### From repeated answers to meaning clusters

At this point, the important question is not whether the sampled answers use exactly the same words. The important question is whether they mean the same thing. The entailment model is used to group answers that are semantically equivalent, so that paraphrases are treated as the same answer cluster.

Semantic entropy is high when the samples are spread across several meaning clusters. It is low when most samples fall into one cluster.

## 5. Compute semantic entropy for each factoid

We now combine the central steps of the semantic entropy pipeline.

For each factoid, the function below should:

1. generate `Q` questions about the factoid;
2. generate `M` answers to each question;
3. use the entailment model to group semantically equivalent answers;
4. compute semantic entropy over the answer clusters for each question;
5. average these entropy values to obtain one semantic entropy score for the factoid.

Intuitively, low entropy means that the model repeatedly gives answers with the same meaning. High entropy means that the model gives answers with several different meanings, suggesting semantic uncertainty.

### Question 5: Complete the combined function
Fill in the missing parts of `calculate_factoid_entropy(...)` below. You may add print statements to make the intermediate steps easier to inspect. Clear diagnostic output can be especially useful here, because semantic entropy is easier to understand when you can see the generated questions, sampled answers, entailment clusters, and entropy scores.

You may use generative AI to help format diagnostic print statements, but make sure you understand the logic of the function you employ.

In [ ]:
def calculate_factoid_entropy(factoid, original_answer, Q, M, model_name, max_tokens, client, entailment_model, add_sub_question=False, verbose=False, enable_sleep=True):
    '''
    Combines step 6 - 7 - 8 of the pipeline for a single factoid:
    - Generates Q questions.
    - Generates M answers per question.
    - Calculates entailment of M answers immediately to buffer API rate limits.
    - Calculates semantic entropy of subquestion based on entailment
    - Computes average semantic entropy for the factoid.
    
    Returns:
        - final_semantic_entropy (float)
        - factoid_to_questions (dict)
        - question_to_answers (dict)
        - factoid_to_sub_scores (dict), sub_scores are the semantic entropies of a factoid subquestion
    '''    
    sub_questions =                                                  # TO DO: generate questions
    
    factoid_to_questions = {factoid: sub_questions}
    question_to_answers = {}
    sub_entropies = {} 
    semantic_entropies = [] 
    
    if verbose:
        print("\n" + "="*80)
        print("FACTOID ENTROPY ANALYSIS")
        print("-" * 80)
        print(f"Original Answer: '{original_answer}'\n")
        print(f"Target Factoid : '{factoid}'\n")
        print("Generated Subquestions:")
        for i, sq in enumerate(sub_questions, 1):
            print(f"  ({i}) {sq}")

    for i, sq in enumerate(sub_questions, 1):
        answers =                                                    # TO DO: generate M answers              
        question_to_answers[...] = ...                               # TO DO
        
        sub_answers = list(answers) 
        sub_answers.append(...)                                      # TO DO
        
        if add_sub_question:
            entailment_inputs = [f"{sq}\n{ans}" for ans in sub_answers]
        else:
            entailment_inputs = sub_answers
            
        # Entailment timer, to keep track of how long entailment takes
        start_time = time.time()
        semantic_ids =                                               # TO DO
        end_time = time.time()
        entailment_duration = end_time - start_time
        
        sub_semantic_entropy =                                       # TO DO call cluster_assignment_entropy(..)
        semantic_entropies.append(sub_semantic_entropy)
        sub_entropies[sq] = sub_semantic_entropy # store sub score per subquestion
        
        if verbose:
            print("\n" + "-"*80)
            print(f"({i}) Subquestion: {sq}\n")
            print("Generated Answers (including original factoid):")
            for j, ans in enumerate(sub_answers, 1):
                label = "[Factoid]" if j == len(sub_answers) else f"[{j}]"
                print(f"  {label} {ans}")
            
            print("\nSemantic Clusters:")
            print(f"  IDs: {semantic_ids}")
            
            print("\nEntropy Analysis:")
            print(f"  Semantic Entropy : {sub_semantic_entropy:.6f}")
            print(f"  Semantic Confidence: {math.exp(-sub_semantic_entropy):.6f}")
            # Print the timer
            print(f"  ⏱️ Entailment Time: {entailment_duration:.2f} seconds")

    if len(semantic_entropies) > 0:
        final_semantic_entropy =                                     # TO DO calculate average semantic entropy of factoid
    else:
        final_semantic_entropy = 0.0
        
    factoid_to_sub_scores = {factoid: sub_entropies} # store the sub entropies per factiod as well

    if verbose:
        print("\n" + "="*80)
        print(f"FACTOID SUMMARY: '{factoid}'")
        print("-" * 80)
        print(f"  Average Semantic Entropy : {final_semantic_entropy:.6f}")
        print(f"  Final Semantic Confidence: {math.exp(-final_semantic_entropy):.6f}")
        print("="*80)

    return final_semantic_entropy, factoid_to_questions, question_to_answers, factoid_to_sub_scores

### Optional Support: Example Prompts

If you were stuck on Question 3 or Question 4, you may use the example prompts below. These prompts are not necessarily perfect, but they provide a reasonable starting point for the question generation and repeated answer steps.

Even if you use them, inspect the outputs critically. Prompt quality has a direct effect on the quality of the semantic entropy estimate.

In [ ]:
# for the generate_Q_questions(...) function

    user_prompt = f"""Based on the Original Text, generate exactly {Q} different questions that are directly answered by the provided factoid.
    
    Original Text: "{original_answer}"
    
    Factoid to target: "{factoid}"
    
    Rules:
    1. Generate exactly {Q} questions.
    2. Vary the phrasing of the questions.
    3. Be specific: ALWAYS include the names of the objects, people, or places in your questions so they can be understood without seeing the original text.
    4. Do not ask questions about information that is NOT in the factoid.
    5. Respond ONLY with a valid JSON array of strings. 

    Example Input:
    "Marie Curie moved to Paris for her studies"
    
    Example Output:
    ["Why did Marie Curie move to Paris?", "Where did Marie Curie move to for her studies?"]
    """

# for the generate_M_answers(...) function:
    user_prompt = f"""Question: {question}
    
    Rules:
    1. Do NOT answer in a full sentence.
    2. Answer with as few words as possible (e.g., only a name, place, number, or thing).
    3. If multiple names/places/things are applicable, provide a EXACTLY 1 item.
    4. Provide ONLY the answer and absolutely nothing else.
    """

## 6. Colour-code the original answer

The final step maps factoid-level uncertainty back onto the original answer.

For each sentence part, we collect the semantic entropy scores of the factoids extracted from that part. We then convert entropy into **semantic confidence** and use this confidence to colour the sentence part. Sentence parts with high confidence are shown closer to green, while sentence parts with higher uncertainty are shown closer to red.

Only start this step once the previous functions run successfully. This part may take some time, because it repeatedly queries the language model and runs entailment comparisons.

Read the function description below, then run the following cells to calculate the entropy scores and display the colour-coded answer.

In [ ]:
def color_code_answer(original_answer, sentence_parts, sentence_part_to_factoids, factoid_to_entropy):
    """
    Reconstructs the original answer, applying a color gradient to each sentence part 
    based on the average semantic confidence of its associated factoids.
    Unscored or skipped text remains uncolored.
    """
    colored_answer = original_answer

    for s_part in sentence_parts:
        factoids = sentence_part_to_factoids.get(s_part, [])
        
        # Print warning if a sentence part has no extracted factoids
        if not factoids:
            print(f"⚠️ Notice: '{s_part[:30]}...' -> Skipped (No factoids extracted).")
            continue

        confidences = []
        for f in factoids:
            # calculate semantic confidence if a factoid has a semantic entropy related to it
            if f in factoid_to_entropy:
                entropy = factoid_to_entropy[f]
                confidence = math.exp(-entropy)
                confidences.append(confidence)
            else:
                # print if a specific factoid missed the entropy calculation
                print(f"⚠️ Notice: Factoid '{f[:30]}...' -> Missing entropy score.")
        
        if len(confidences) == 0:
            # print if no confidences could be calculated at all
            print(f"⚠️ Notice: '{s_part[:30]}...' -> Skipped (Could not compute final confidence).")

        else:
            avg_confidence = sum(confidences) / len(confidences)
            bg_hex = prob_to_color(avg_confidence)
            colored_s_part = colored(s_part, bg_hex)
            colored_answer = colored_answer.replace(s_part, colored_s_part, 1)

    return colored_answer

In [ ]:
Q_QUESTIONS = 2         # Number of questions to generate per factoid
M_ANSWERS = 3           # Number of independent answers to generate per question
ENABLE_SLEEP = True     # Turn on the 20 second rate limit buffer
VERBOSE = True          # Turn on the detailed math, timer, and diagnostic prints
ADD_SUB_QUESTION = False # Prepends the subquestion to the subanswers for the entailment model

print("\n" + "=" * 80)
print(f"CALCULATING SEMANTIC ENTROPY (Q={Q_QUESTIONS}, M={M_ANSWERS})")
print("=" * 80)

# Initialize master dictionaries to hold all data across all factoids
master_factoid_to_questions = {}
master_question_to_answers = {}
master_factoid_scores = {}
master_final_entropies = {}

# Loop through every factoid we extracted before
for i, factoid in enumerate(factoids, 1):
    print(f"\n⚙️ Processing Factoid {i}/{len(factoids)}...")
    
    # Run the step 6 - 7 - 8 function
    final_entropy, f_to_q, q_to_a, f_to_scores = calculate_factoid_entropy(
        factoid=factoid,
        original_answer=answer,
        Q=Q_QUESTIONS,
        M=M_ANSWERS,
        model_name=model_name,
        max_tokens=max_tokens,
        client=client,
        entailment_model=entailment_model,
        add_sub_question=ADD_SUB_QUESTION,
        verbose=VERBOSE,
        enable_sleep=ENABLE_SLEEP
    )
    
    # Update our master dictionaries with the results from this factoid
    master_factoid_to_questions.update(f_to_q)
    master_question_to_answers.update(q_to_a)
    master_factoid_scores.update(f_to_scores)
    master_final_entropies[factoid] = final_entropy


### Question 6: Interpret the Colour-Coded Answer

Run the final cell to display the answer after applying the semantic entropy pipeline. Then reflect on the result.

In your answer, address the following points:

- Do the colours match your own judgement from Question 1? Explain where they agree or disagree.
- Are any parts of the answer left uncoloured? If so, trace this back through the pipeline. Was the sentence part not extracted, were no factoids found, or did a later step fail?
- What are the main limitations of this semantic entropy pipeline?
- Which parts of the pipeline work well, and which parts would you improve?
- Propose at least one concrete improvement. 

Write your response in the cell below.

In [ ]:
# TO DO: write your answer to question 6 here


In [ ]:
# color coding!
print("\n" + "=" * 80)
print("FINAL RESULT: SEMANTIC ENTROPY COLOR MAP")
print("=" * 80)
print("Legend: [GREEN] = High Confidence | [RED] = High Uncertainty (Hallucination Risk)\n")

# Run the color coding function
final_colored_output = color_code_answer(
    original_answer=answer, 
    sentence_parts=sentence_parts,           
    sentence_part_to_factoids=s_part_to_factoids,
    factoid_to_entropy=master_final_entropies     
)

# Print the final reconstructed string!
print(final_colored_output)
print("\n" + "=" * 80)